# MDCT

In [1]:
import numpy as np
from scipy.io import wavfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
import os
import hashlib
import numpy as np

In [2]:
def mdct4(x):
    N = x.shape[0]
    if N%4 != 0:
        raise ValueError("MDCT4 only defined for vectors of length multiple of four.")
    M = N // 2
    N4 = N // 4
    
    rot = np.roll(x, N4)
    rot[:N4] = -rot[:N4]
    t = np.arange(0, N4)
    w = np.exp(-1j*2*np.pi*(t + 1./8.) / N)
    c = np.take(rot,2*t) - np.take(rot, N-2*t-1)         - 1j * (np.take(rot, M+2*t) - np.take(rot,M-2*t-1))
    c = (2./np.sqrt(N)) * w * np.fft.fft(0.5 * c * w, N4)
    y = np.zeros(M)
    y[2*t] = np.real(c[t])
    y[M-2*t-1] = -np.imag(c[t])
    return y

In [3]:
def imdct4(x):
    N = x.shape[0]
    if N%2 != 0:
        raise ValueError("iMDCT4 only defined for even-length vectors.")
    M = N // 2
    N2 = N*2
    
    t = np.arange(0,M)
    w = np.exp(-1j*2*np.pi*(t + 1./8.) / N2)
    c = np.take(x,2*t) + 1j * np.take(x,N-2*t-1)
    c = 0.5 * w * c
    c = np.fft.fft(c,M)
    c = ((8 / np.sqrt(N2))*w)*c
    
    rot = np.zeros(N2)
    
    rot[2*t] = np.real(c[t])
    rot[N+2*t] = np.imag(c[t])
    
    t = np.arange(1,N2,2)
    rot[t] = -rot[N2-t-1]
    
    t = np.arange(0,3*M)
    y = np.zeros(N2)
    y[t] = rot[t+M]
    t = np.arange(3*M,N2)
    y[t] = -rot[t-3*M]
    return y

sign flip по ключу

In [4]:
KEY_PATH = "./key.bin"

def load_key_bytes(path):
    with open(path, "rb") as f:
        return f.read()

def _seed_from_key(key_bytes, salt):
    h = hashlib.sha256()
    h.update(key_bytes)
    h.update(str(salt).encode("utf-8"))
    return int.from_bytes(h.digest()[:8], "big", signed=False)

def sign_mask_vector(size, key_bytes, salt=""):
    rng = np.random.default_rng(_seed_from_key(key_bytes, salt))
    return rng.choice(np.array([-1.0, 1.0]), size=size).astype(np.float64)

def sign_mask_blocks(n_samples, block_size, key_bytes, salt=""):
    rng = np.random.default_rng(_seed_from_key(key_bytes, salt))
    n_blocks = (n_samples + block_size - 1) // block_size
    signs = rng.choice(np.array([-1.0, 1.0]), size=n_blocks).astype(np.float64)
    return np.repeat(signs, block_size)[:n_samples]

key_bytes = load_key_bytes(KEY_PATH)

In [5]:
def mdct_signflip(Y, key_bytes):
    mask = sign_mask_vector(len(Y), key_bytes, salt=f"mdct_sign_{len(Y)}")
    return Y * mask

#### main

In [6]:
M = 1024
N = 2 * M
hop = M

In [7]:
def process_mdct_blocks(data, modify_func=None):
    n = np.arange(N)
    window = np.sin(np.pi / N * (n + 0.5))

    pad = (N - (len(data) % hop)) % hop
    data = np.pad(data, (0, pad))

    output = np.zeros_like(data)

    for start in range(0, len(data) - N + 1, hop):
        frame = data[start:start+N] * window
        Y = mdct4(frame)

        if modify_func is not None:
            Y = modify_func(Y)

        Z = imdct4(Y)
        output[start:start+N] += Z * window

    return output

=================

In [8]:
DATASET_DIR = "../../dataset/test_sounds"

OUT_ENC_INVERT_DIR = "./assets/encrypted/invert"
OUT_DEC_INVERT_DIR = "./assets/decrypted/invert"

OUT_ENC_SHUFFLE_DIR = "./assets/encrypted/shuffle"
OUT_DEC_SHUFFLE_DIR = "./assets/decrypted/shuffle"

OUT_ENC_SIGNFLIP_DIR = "./assets/encrypted/signflip"
OUT_DEC_SIGNFLIP_DIR = "./assets/decrypted/signflip"

os.makedirs(OUT_ENC_INVERT_DIR, exist_ok=True)
os.makedirs(OUT_DEC_INVERT_DIR, exist_ok=True)
os.makedirs(OUT_ENC_SHUFFLE_DIR, exist_ok=True)
os.makedirs(OUT_DEC_SHUFFLE_DIR, exist_ok=True)
os.makedirs(OUT_ENC_SIGNFLIP_DIR, exist_ok=True)
os.makedirs(OUT_DEC_SIGNFLIP_DIR, exist_ok=True)

N_FILES = 23

perm = np.random.permutation(M)
inv_perm = np.argsort(perm)

def invert(Y):
    return Y[::-1]

def shuffle(Y):
    return Y[perm]

def unshuffle(Y):
    return Y[inv_perm]

def write_wav(path, fs, sig):
    sig = sig.astype(np.float64)
    m = np.max(np.abs(sig)) + 1e-12
    sig = sig / m
    wavfile.write(path, fs, (sig * 32767).astype(np.int16))

for i in range(1, N_FILES + 1):
    in_path = os.path.join(DATASET_DIR, f"test_sound{i:1d}.wav")

    fs, data = wavfile.read(in_path)

    if data.ndim > 1:
        data = data[:, 0]

    x = data.astype(np.float64)

    inv_audio = process_mdct_blocks(x, invert)
    inv_dec = process_mdct_blocks(inv_audio, invert)

    write_wav(os.path.join(OUT_ENC_INVERT_DIR, f"test_sound{i:1d}_invert_enc.wav"), fs, inv_audio)
    write_wav(os.path.join(OUT_DEC_INVERT_DIR, f"test_sound{i:1d}_invert_dec.wav"), fs, inv_dec)

    shuf_audio = process_mdct_blocks(x, shuffle)
    shuf_dec = process_mdct_blocks(shuf_audio, unshuffle)

    write_wav(os.path.join(OUT_ENC_SHUFFLE_DIR, f"test_sound{i:1d}_shuffle_enc.wav"), fs, shuf_audio)
    write_wav(os.path.join(OUT_DEC_SHUFFLE_DIR, f"test_sound{i:1d}_shuffle_dec.wav"), fs, shuf_dec)

    sign_audio = process_mdct_blocks(x, lambda Y: mdct_signflip(Y, key_bytes))
    sign_dec = process_mdct_blocks(sign_audio, lambda Y: mdct_signflip(Y, key_bytes))

    write_wav(os.path.join(OUT_ENC_SIGNFLIP_DIR, f"test_sound{i:1d}_signflip_enc.wav"), fs, sign_audio)
    write_wav(os.path.join(OUT_DEC_SIGNFLIP_DIR, f"test_sound{i:1d}_signflip_dec.wav"), fs, sign_dec)